In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window
from datetime import datetime as dt
import uuid

VOLUMES = {
    "bronze":           "/Volumes/workspace/olist/bronze",
    "silver":           "/Volumes/workspace/olist/silver",
    "dead_letter":      "/Volumes/workspace/olist/dead_letter"
}

EXECUCAO_ID = str(uuid.uuid4())

print(f"Configurações carregadas!")
print(f"Execução ID: {EXECUCAO_ID}")

# Lê o Silver gravado na Sprint 3
df_silver = spark.read.format("delta").load(f"{VOLUMES['silver']}/orders_enriched")

print(f"Registros lidos do Silver: {df_silver.count()}")
print(f"Colunas: {len(df_silver.columns)}")

In [0]:
import time
from pyspark.sql.types import (StructType, StructField, StringType, 
                                LongType, DoubleType, TimestampType)
from datetime import datetime as dt, date

LOG_PATH   = "/Volumes/workspace/olist/bronze/_log"
LOG_SCHEMA = StructType([
    StructField("execucao_id",        StringType(),    nullable=False),
    StructField("tabela",             StringType(),    nullable=False),
    StructField("status",             StringType(),    nullable=False),
    StructField("registros_lidos",    LongType(),      nullable=True),
    StructField("registros_gravados", LongType(),      nullable=True),
    StructField("divergencia_pct",    DoubleType(),    nullable=True),
    StructField("tempo_segundos",     DoubleType(),    nullable=True),
    StructField("mensagem",           StringType(),    nullable=True),
    StructField("timestamp",          TimestampType(), nullable=False)
])

def gravar_log(execucao_id, tabela, status, registros_lidos=None,
               registros_gravados=None, divergencia_pct=None,
               tempo_segundos=None, mensagem=None):

    linha = [{
        "execucao_id":        execucao_id,
        "tabela":             tabela,
        "status":             status,
        "registros_lidos":    registros_lidos,
        "registros_gravados": registros_gravados,
        "divergencia_pct":    divergencia_pct,
        "tempo_segundos":     tempo_segundos,
        "mensagem":           mensagem,
        "timestamp":          dt.now()
    }]

    spark.createDataFrame(linha, schema=LOG_SCHEMA) \
        .write \
        .format("delta") \
        .mode("append") \
        .save(LOG_PATH)

def processar_silver(execucao_id: str):
    inicio = time.time()
    nome   = "orders_enriched"

    print(f"\n{'='*50}")
    print(f"Processando Silver: {nome}")
    print(f"{'='*50}")

    try:
        # Lê o Silver gravado
        df = spark.read.format("delta").load(f"{VOLUMES['silver']}/{nome}")
        registros_lidos = df.count()
        print(f"  Lidos: {registros_lidos} registros")

        # Consolida todos os checks
        problemas = []

        # Check 1 — divergência contra Bronze
        registros_bronze = spark.read.format("delta") \
            .load(f"{VOLUMES['bronze']}/order_items").count()
        divergencia = abs(registros_lidos - registros_bronze)
        if divergencia > 0:
            problemas.append(f"Divergência contra Bronze: {divergencia} registros")

        # Check 2 — órfãos pós-join
        orfaos = df.filter(F.col("order_status").isNull()).count()
        if orfaos > 0:
            problemas.append(f"Órfãos pós-join: {orfaos} registros")

        # Check 3 — duplicatas na chave primária
        duplicatas = registros_lidos - df.select("order_id", "order_item_id").distinct().count()
        if duplicatas > 0:
            problemas.append(f"Duplicatas na chave primária: {duplicatas}")

        # Check 4 — nulos nas colunas calculadas
        for coluna in ["valor_total_pedido", "rank_item_pedido", "ltv_acumulado"]:
            nulos = df.filter(F.col(coluna).isNull()).count()
            if nulos > 0:
                problemas.append(f"Nulos em {coluna}: {nulos}")

        # Check 5 — valores impossíveis
        if df.filter(F.col("valor_total_pedido") <= 0).count() > 0:
            problemas.append("valor_total_pedido com zero ou negativo")
        if df.filter(F.col("ltv_acumulado") <= 0).count() > 0:
            problemas.append("ltv_acumulado com zero ou negativo")

        # Check 6 — rank mínimo
        rank_min = df.agg(F.min("rank_item_pedido")).collect()[0][0]
        if rank_min != 1:
            problemas.append(f"Rank mínimo inválido: {rank_min}")

        # Decisão final
        tempo = round(time.time() - inicio, 2)

        if problemas:
            motivo = " | ".join(problemas)
            status = "FAILED"
            mensagem = f"Falha nas validações: {motivo}"
            print(f"  FALHA: {motivo}")
        else:
            status = "SUCCESS"
            mensagem = f"Processado com sucesso em {tempo}s"
            print(f"  SUCCESS — {registros_lidos} registros validados em {tempo}s")

        gravar_log(
            execucao_id        = execucao_id,
            tabela             = nome,
            status             = status,
            registros_lidos    = registros_lidos,
            registros_gravados = registros_lidos if status == "SUCCESS" else None,
            tempo_segundos     = tempo,
            mensagem           = mensagem
        )

    except Exception as e:
        tempo = round(time.time() - inicio, 2)
        gravar_log(
            execucao_id    = execucao_id,
            tabela         = nome,
            status         = "FAILED",
            tempo_segundos = tempo,
            mensagem       = f"Erro inesperado: {str(e)}"
        )
        print(f"  ERRO INESPERADO: {e}")

print("Orquestrador Silver carregado!")

In [0]:
# Executa o orquestrador
gravar_log(
    execucao_id = EXECUCAO_ID,
    tabela      = "PIPELINE_SILVER",
    status      = "STARTED",
    mensagem    = "Início da blindagem do Silver"
)

processar_silver(EXECUCAO_ID)

# Resumo da execução
df_resumo = spark.read \
    .format("delta") \
    .load(LOG_PATH) \
    .filter(F.col("execucao_id") == EXECUCAO_ID) \
    .filter(F.col("tabela") != "PIPELINE_SILVER")

print(f"\n{'='*50}")
print(f"RESUMO DA EXECUÇÃO — {EXECUCAO_ID}")
print(f"{'='*50}")

df_resumo.select(
    "tabela",
    "status",
    "registros_lidos",
    "registros_gravados",
    "tempo_segundos",
    "mensagem"
).show(truncate=False)

# Registra fim
falhas   = df_resumo.filter(F.col("status") == "FAILED").count()
sucessos = df_resumo.filter(F.col("status") == "SUCCESS").count()

gravar_log(
    execucao_id = EXECUCAO_ID,
    tabela      = "PIPELINE_SILVER",
    status      = "FINISHED" if falhas == 0 else "FINISHED_WITH_ERRORS",
    mensagem    = f"Concluído — SUCCESS: {sucessos} | FAILED: {falhas}"
)

print(f"\nSUCCESS: {sucessos} | FAILED: {falhas}")
print(f"Pipeline Silver finalizado!")